# 컴퓨터비전 부트캠프 · 2일차 실습
## 성능 개선과 전이학습

부경대학교 교내 컴퓨터비전 부트캠프 · 2026. 8. 4.
김한울 (서울과학기술대학교 인공지능응용학과)

---

어제 만든 baseline(검증 정확도 약 0.68)을 **단계적으로 끌어올려** 봅니다.
한 번에 하나씩만 바꾸고, 매번 결과를 표에 기록합니다.

| STEP | 하는 일 | 기대 결과 |
|---|---|---|
| STEP 0 | 준비 — 어제 코드 요약본 실행 | baseline 재현 |
| STEP 1 | 데이터 증강 적용 | +5 ~ 8%p |
| STEP 2 | learning rate 스케줄러 | +1 ~ 3%p |
| STEP 3 | ResNet18 Feature Extractor | 0.84 내외 |
| STEP 4 | ResNet18 Fine-tuning | 0.93 내외 |
| STEP 5 | 결과 비교와 정리 | 실험 기록표 완성 |

> **먼저 할 일** — `런타임` → `런타임 유형 변경` → `T4 GPU`

> 이 노트북에서 얻은 결론을 오후 해커톤에 그대로 가져가면 됩니다.

---
## STEP 0. 준비

어제 만든 것을 함수로 정리해 두었습니다. 내용은 어제와 같으니 그대로 실행하세요.

In [ ]:
import random, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치 :", device)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

plt.rcParams["figure.dpi"] = 110

### 그래프에 한글 쓰기 (Colab)

Colab 의 matplotlib 에는 한글 폰트가 없어 그래프 안의 한글이 네모로 깨집니다.
아래 셀을 한 번 실행해 두면 해결됩니다. (약 20초)

In [ ]:
# Colab 한글 폰트 설정 — 실패해도 실습에는 지장이 없습니다
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run("apt-get install -y -qq fonts-nanum", shell=True, check=True)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 설정을 건너뜁니다 :", e)
plt.rcParams["axes.unicode_minus"] = False

### 데이터셋과 학습 함수

`make_loaders()` 는 transform 두 개(학습용·평가용)를 받아 DataLoader 세 개를 돌려줍니다.
증강을 실험할 때 이 함수만 다시 부르면 됩니다.

In [ ]:
def make_loaders(transform_train, transform_eval, batch_size=128):
    full_train = datasets.CIFAR10(root="./data", train=True, download=True,
                                  transform=transform_train)
    full_eval  = datasets.CIFAR10(root="./data", train=True, download=True,
                                  transform=transform_eval)
    test_set   = datasets.CIFAR10(root="./data", train=False, download=True,
                                  transform=transform_eval)

    g = torch.Generator().manual_seed(SEED)
    train_idx, val_idx = random_split(range(50000), [45000, 5000], generator=g)

    train_set = torch.utils.data.Subset(full_train, list(train_idx))
    val_set   = torch.utils.data.Subset(full_eval,  list(val_idx))   # val 은 증강 없음

    return (DataLoader(train_set, batch_size=batch_size, shuffle=True,
                       num_workers=2, pin_memory=True),
            DataLoader(val_set, batch_size=256, shuffle=False,
                       num_workers=2, pin_memory=True),
            DataLoader(test_set, batch_size=256, shuffle=False,
                       num_workers=2, pin_memory=True))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    tot_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item() * labels.size(0)
        correct  += (outputs.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return tot_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    tot_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        tot_loss += criterion(outputs, labels).item() * labels.size(0)
        correct  += (outputs.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return tot_loss / total, correct / total

### 실험 하나를 끝까지 돌리는 함수

`fit()` 은 학습 → 검증을 epoch 만큼 반복하고, 가장 좋았던 검증 정확도를 돌려줍니다.
결과는 전역 리스트 `RESULTS` 에 자동으로 쌓입니다.

In [ ]:
RESULTS = []

def fit(name, model, train_loader, val_loader, epochs=10, lr=1e-3,
        weight_decay=1e-2, scheduler_name=None, param_groups=None, note=""):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    params = param_groups if param_groups is not None else model.parameters()
    optimizer = optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    scheduler = None
    if scheduler_name == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif scheduler_name == "step":
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3),
                                              gamma=0.1)

    hist = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best_acc, best_state = 0.0, None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()

        hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
        hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)

        if va_acc > best_acc:
            best_acc = va_acc
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        print(f"  epoch {ep:2d}/{epochs} | train {tr_acc:.4f} | val {va_acc:.4f}")

    elapsed = time.time() - t0
    RESULTS.append({"name": name, "val_acc": best_acc,
                    "epochs": epochs, "sec": elapsed, "note": note})
    print(f"[{name}] 최고 검증 정확도 {best_acc:.4f}  ({elapsed:.0f}초)")
    model.load_state_dict(best_state)
    return model, hist


def show_results():
    print(f"{'실험':<28}{'val acc':>9}{'변화':>9}{'epoch':>7}{'시간(초)':>10}   메모")
    print("-" * 92)
    base = RESULTS[0]["val_acc"] if RESULTS else 0
    for i, r in enumerate(RESULTS):
        delta = "—" if i == 0 else f"{r['val_acc'] - base:+.4f}"
        print(f"{r['name']:<28}{r['val_acc']:>9.4f}{delta:>9}"
              f"{r['epochs']:>7}{r['sec']:>10.0f}   {r['note']}")

### baseline 재현

어제와 같은 SimpleCNN, 증강 없음, 10 epoch. 약 2분 걸립니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),   nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


transform_plain = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_loader, val_loader, test_loader = make_loaders(transform_plain, transform_plain)

torch.manual_seed(SEED)
model_base, hist_base = fit("0. baseline (증강 없음)", SimpleCNN(),
                            train_loader, val_loader, epochs=10,
                            note="어제와 동일")

---
## STEP 1. 데이터 증강

### 1-1. 증강 transform 만들기

**학습용에만** 증강을 넣고, 평가용은 그대로 둡니다.

| 기법 | 코드 | 효과 |
|---|---|---|
| 랜덤 크롭 | `transforms.RandomCrop(32, padding=4)` | 위치 변화에 강해짐 |
| 좌우 반전 | `transforms.RandomHorizontalFlip()` | 좌우 대칭 사물에 효과적 |

> `RandomCrop` 과 `RandomHorizontalFlip` 은 PIL 이미지에 적용되므로
> **`ToTensor()` 앞**에 와야 합니다.

In [ ]:
# TODO: RandomCrop(32, padding=4) 과 RandomHorizontalFlip() 을
#       ToTensor() 앞에 추가한 transform 을 만드세요.
transform_aug = transforms.Compose([
    # 여기를 채우세요 (증강 2줄)
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_loader_aug, val_loader, test_loader = make_loaders(transform_aug, transform_plain)
print("증강 적용 완료")

### 1-2. 증강 결과를 눈으로 확인

같은 이미지가 매번 다르게 나오는지 봅니다.

In [ ]:
def show_aug_samples(dataset, index=7, n=8):
    fig, axes = plt.subplots(1, n, figsize=(13, 2.2))
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std  = torch.tensor(CIFAR10_STD).view(3, 1, 1)
    for k, ax in enumerate(axes):
        img, label = dataset[index]                  # 꺼낼 때마다 증강이 새로 적용된다
        ax.imshow((img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy())
        ax.set_title("원본 기준" if k == 0 else f"증강 {k}", fontsize=9)
        ax.axis("off")
    plt.suptitle(f"같은 이미지({CLASSES[label]})에 증강을 반복 적용한 결과", fontsize=12)
    plt.tight_layout(); plt.show()


show_aug_samples(train_loader_aug.dataset)

### 1-3. 증강을 넣고 다시 학습

증강을 쓰면 모델이 데이터를 외우기 어려워지므로 **epoch 를 늘려도 과적합이 덜합니다.**
여기서는 비교를 위해 일단 같은 10 epoch 으로 돌립니다.

In [ ]:
torch.manual_seed(SEED)
model_aug, hist_aug = fit("1. + 데이터 증강", SimpleCNN(),
                          train_loader_aug, val_loader, epochs=10,
                          note="RandomCrop + HorizontalFlip")
show_results()

### 1-4. 학습 곡선 비교

증강 전후의 **train 과 validation 격차**가 어떻게 달라졌는지 봅니다.
격차가 좁아졌다면 과적합이 줄어든 것입니다.

In [ ]:
def plot_compare(hists, labels, key="val_acc", title="Validation accuracy"):
    plt.figure(figsize=(9, 4))
    colors = ["#156082", "#E97132", "#0F9ED5", "#196B24", "#A02B93"]
    for h, lb, c in zip(hists, labels, colors):
        plt.plot(range(1, len(h[key]) + 1), h[key], "-o", label=lb, color=c)
    plt.xlabel("epoch"); plt.ylabel(key); plt.title(title)
    plt.legend(); plt.grid(color="#EEEEEE"); plt.tight_layout(); plt.show()


plot_compare([hist_base, hist_aug], ["baseline", "+ augmentation"])

for name, h in [("baseline", hist_base), ("augmented", hist_aug)]:
    gap = h["train_acc"][-1] - h["val_acc"][-1]
    print(f"{name:<12} train {h['train_acc'][-1]:.4f} - val {h['val_acc'][-1]:.4f} "
          f"= gap {gap:.4f}")

---
## STEP 2. learning rate 스케줄러

후반으로 갈수록 보폭을 줄이면 더 낮은 지점에 안착합니다.
증강을 켠 상태에서 `CosineAnnealingLR` 을 추가하고, epoch 도 20으로 늘립니다.

```python
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
...
scheduler.step()     # epoch 이 끝날 때마다
```

`fit()` 함수에 이미 들어 있으므로 `scheduler_name="cosine"` 만 넘기면 됩니다.

In [ ]:
torch.manual_seed(SEED)
# TODO: fit() 에 scheduler_name="cosine" 을 넘기고 epochs 를 20 으로 설정하세요.
model_sched, hist_sched = fit("2. + CosineAnnealingLR", SimpleCNN(),
                              train_loader_aug, val_loader,
                              epochs=10,              # <- 20 으로 고치세요
                              scheduler_name=None,    # <- "cosine" 으로 고치세요
                              note="증강 유지, epoch 10 -> 20")
show_results()

---
## STEP 3. 전이학습 ① Feature Extractor

여기서부터 방식이 달라집니다. 직접 만든 SimpleCNN 대신
**ImageNet 130만 장으로 이미 학습된 ResNet18** 을 가져옵니다.

### 3-1. 입력 규격 맞추기

ResNet 은 224×224 입력과 ImageNet 정규화 값을 기대합니다.

> Colab 무료 GPU 기준 224는 다소 무겁습니다. 이 실습에서는 **112** 로 타협합니다.
> 성능은 조금 낮아지지만 학습 시간이 1/4 로 줄어듭니다. 해커톤에서 시간이 남으면 224로 올려 보세요.

In [ ]:
IMG_SIZE = 112

# TODO: 학습용 transform 을 만드세요.
#       Resize(IMG_SIZE) -> RandomCrop(IMG_SIZE, padding=IMG_SIZE//8)
#       -> RandomHorizontalFlip() -> ToTensor() -> Normalize(IMAGENET_MEAN, IMAGENET_STD)
transform_tf_train = transforms.Compose([
    # 여기를 채우세요
])

# TODO: 평가용 transform — 증강 없이 Resize -> ToTensor -> Normalize
transform_tf_eval = transforms.Compose([
    # 여기를 채우세요
])

tf_train_loader, tf_val_loader, tf_test_loader = make_loaders(
    transform_tf_train, transform_tf_eval, batch_size=64)

images, labels = next(iter(tf_train_loader))
print("배치 모양 :", images.shape)   # (64, 3, 112, 112) 이어야 한다

### 3-2. backbone 을 얼리고 분류기만 새로 학습

1. 사전학습 가중치와 함께 ResNet18 을 불러온다
2. 모든 파라미터를 `requires_grad = False` 로 고정한다
3. `fc` 를 10 클래스로 교체한다 (새로 만든 층은 자동으로 학습 대상)

In [ ]:
def build_resnet18_feature_extractor():
    # TODO 1: 사전학습 가중치와 함께 resnet18 을 불러오세요.
    #         힌트: models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model = None   # 여기를 채우세요

    # TODO 2: 모든 파라미터를 고정하세요 (requires_grad = False)
    # 여기를 채우세요

    # TODO 3: fc 를 10 클래스짜리 Linear 로 교체하세요.
    #         힌트: nn.Linear(model.fc.in_features, 10)
    # 여기를 채우세요

    return model


m = build_resnet18_feature_extractor()
trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
total     = sum(p.numel() for p in m.parameters())
print(f"학습 대상 {trainable:,} / 전체 {total:,}  ({trainable / total:.2%})")

### 3-3. 학습

학습 대상 파라미터가 5천 개뿐이라 빠릅니다. 5 epoch 이면 충분합니다.

In [ ]:
torch.manual_seed(SEED)
model_fe, hist_fe = fit("3. ResNet18 Feature Extractor",
                        build_resnet18_feature_extractor(),
                        tf_train_loader, tf_val_loader,
                        epochs=5, lr=1e-3,
                        note=f"backbone 고정, {IMG_SIZE}px")
show_results()

---
## STEP 4. 전이학습 ② Fine-tuning

이번에는 backbone 도 함께 학습합니다.
단, **backbone 에는 작은 learning rate** 를 줘야 애써 배운 특징이 망가지지 않습니다.

| 부분 | learning rate | 이유 |
|---|---|---|
| backbone (Conv 층들) | `1e-4` | 이미 잘 학습되어 있다. 살짝만 조정 |
| 새 분류층 `fc` | `1e-3` | 가중치가 무작위다. 빠르게 배워야 한다 |

In [ ]:
def build_resnet18_finetune():
    # TODO: 사전학습 resnet18 을 불러오고 fc 만 10 클래스로 교체하세요.
    #       (이번에는 파라미터를 고정하지 않습니다)
    model = None   # 여기를 채우세요
    # 여기에 fc 교체 코드
    return model


model_ft_raw = build_resnet18_finetune()

# TODO: fc 를 제외한 파라미터를 모아 backbone_params 를 만드세요.
#       힌트: [p for name, p in model.named_parameters() if not name.startswith("fc")]
backbone_params = None   # 여기를 채우세요

# TODO: backbone 은 lr=1e-4, fc 는 lr=1e-3 인 param_groups 를 만드세요.
param_groups = [
    # {"params": ..., "lr": ...},
    # {"params": ..., "lr": ...},
]

torch.manual_seed(SEED)
model_ft, hist_ft = fit("4. ResNet18 Fine-tuning", model_ft_raw,
                        tf_train_loader, tf_val_loader,
                        epochs=5, scheduler_name="cosine",
                        param_groups=param_groups,
                        note=f"backbone lr=1e-4, fc lr=1e-3, {IMG_SIZE}px")
show_results()

---
## STEP 5. 결과 정리

### 5-1. 실험 기록표

In [ ]:
show_results()

plt.figure(figsize=(10, 4))
names = [r["name"] for r in RESULTS]
accs  = [r["val_acc"] for r in RESULTS]
bars = plt.bar(range(len(accs)), accs,
               color=["#156082", "#156082", "#156082", "#0F9ED5", "#E97132"][:len(accs)])
for b, v in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.012, f"{v:.3f}",
             ha="center", fontsize=12, fontweight="bold")
plt.xticks(range(len(names)), [n.split(". ")[-1] for n in names],
           rotation=20, ha="right", fontsize=9)
plt.ylabel("best validation accuracy"); plt.ylim(0, 1.05)
plt.grid(axis="y", color="#EEEEEE"); plt.tight_layout(); plt.show()

### 5-2. 최종 모델의 test 정확도

가장 좋았던 모델 하나만 골라 **test 셋으로 딱 한 번** 측정합니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(model_ft.to(device), tf_test_loader, criterion)
print(f"최종 모델 test accuracy : {test_acc:.4f}")

### 5-3. 아직 무엇이 안 되는가

정확도가 0.93 이 되어도 700장은 여전히 틀립니다. 무엇이 남았는지 봅니다.

In [ ]:
@torch.no_grad()
def confusion(model, loader):
    model.eval()
    cm = np.zeros((10, 10), dtype=int)
    for images, labels in loader:
        preds = model(images.to(device)).argmax(1).cpu().numpy()
        for t, p in zip(labels.numpy(), preds):
            cm[t, p] += 1
    return cm


cm = confusion(model_ft, tf_test_loader)
per_class = cm.diagonal() / cm.sum(axis=1)

print("클래스별 정확도 (낮은 순)")
for i in np.argsort(per_class):
    print(f"  {CLASSES[i]:<12} {per_class[i]:.3f}")

print("\n가장 많이 헷갈린 쌍 TOP 5")
pairs = [(cm[i, j], i, j) for i in range(10) for j in range(10) if i != j]
for c, i, j in sorted(pairs, reverse=True)[:5]:
    print(f"  실제 {CLASSES[i]:<12} → {CLASSES[j]:<12} {c:4d} 장")

### 5-4. 직접 답해 보기

아래 셀을 더블클릭해 채워 보세요. 오후 해커톤의 출발점입니다.

1. 가장 효과가 컸던 변경은 무엇이었나요? 몇 %p 올랐나요?

    →

2. 예상보다 효과가 작았던 것은 무엇이었나요? 왜 그랬을까요?

    →

3. 어제 baseline 에서 가장 헷갈리던 쌍(cat↔dog)은 지금 얼마나 나아졌나요?

    →

4. 해커톤에서 가장 먼저 시도할 것 하나를 고른다면?

    →

---

## 도전 과제

| # | 과제 | 힌트 |
|---|---|---|
| 1 | `IMG_SIZE` 를 224로 올려 보기 | `batch_size` 를 32로 줄여야 할 수 있습니다 |
| 2 | backbone 을 ResNet50 으로 바꾸기 | `models.resnet50(weights=models.ResNet50_Weights.DEFAULT)` |
| 3 | EfficientNet-B0 써 보기 | 마지막 층 이름이 `fc` 가 아니라 `classifier[1]` 입니다 |
| 4 | 증강을 더 강하게 | `ColorJitter`, `RandomErasing` (`ToTensor()` 뒤) |
| 5 | 앙상블 | 서로 다른 모델 2개의 softmax 확률을 평균 |

---

## 마무리 체크리스트

- [ ] 증강을 train 에만 적용했는지 확인했다
- [ ] 증강 전후 train-val 격차가 줄어드는 것을 확인했다
- [ ] Feature Extractor 와 Fine-tuning 의 차이를 코드로 설명할 수 있다
- [ ] backbone 에 작은 learning rate 를 주는 이유를 안다
- [ ] 실험 기록표를 채웠다

오후 해커톤에서 만나요.